##Feature Selection: Exploratory Analysis of the LCBH Data

The LCBH tract file combines ten years of eviction filings (2010–2019) for Chicago's 804 census tracts with American Community Survey (ACS) estimates. Filing years 2010–2012 are matched to the 2006–2010 ACS and 2013–2019 to the 2014–2018 ACS. 

Before modeling, we explored how the filing rate varies over time and across neighborhoods (LCBH_Figures.ipynb). We discovered that filings fell 25% between 2010 (24,390) and 2019 (18,220) after increasing slightly in 2012, and the citywide rate dropped from 4.08 to 2.88 filings per 100 rental units. We also discovered that 13% of tracts averaged at least double the citywide rate, and the 15 highest-rate community areas are all on the South and West Sides.

We chose features by checking each ACS neighborhood measure against the tract filing rate using Spearman correlation.

We kept features regarding the share of Black residents (+0.72), share of White residents (−0.81), poverty rate (+0.63), median rent (−0.53) and the share of Latinx residents (−0.33).

We took away the rental share of housing (+0.02), because it showed no relationship with the filing rate. We also took out LCBH's court-outcome columns, such as completed cases, eviction orders, back rent and attorney representation. This is because they describe what happened to the filings in court, so they are only known after the filings occur and would cause our features to display improperly.

Note: Translated code to Spark using AI such as Microsoft Copilot 

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from IPython.display import display
import pandas as pd
import os

spark = SparkSession.builder.getOrCreate()

if "DATABRICKS_RUNTIME_VERSION" in os.environ:
    DATA_FOLDER = "/Volumes/cs483eviction/raw/lcbh_raw"
else:
    DATA_FOLDER = "data/lcbh_raw"

lcbh = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(DATA_FOLDER + "/lcbh_tract_year.csv")
)

lcbh = (
    lcbh
    .withColumn("median_rent", F.when(F.col("median_rent") > 0, F.col("median_rent")))
    .withColumn("back_rent_median", F.when(F.col("back_rent_median") > 0, F.col("back_rent_median")))
    .withColumn("poverty_rate", F.when(F.col("population_total") > 0,
                                    F.col("population_poverty_below") / F.col("population_total") * 100))
    .withColumn("pct_black", F.when(F.col("population_total") > 0,
                                    F.col("population_race_black") / F.col("population_total") * 100))
    .withColumn("pct_latinx", F.when(F.col("population_total") > 0,
                                    F.col("population_race_latinx") / F.col("population_total") * 100))
    .withColumn("pct_white", F.when(F.col("population_total") > 0,
                                    F.col("population_race_white") / F.col("population_total") * 100))
    .withColumn("renter_share", F.when(F.col("housing_units_total") > 0,
                                    F.col("housing_units_rental") / F.col("housing_units_total") * 100))
    .withColumn("eviction_order_pct", F.when(F.col("eviction_filings_completed") > 0,
                                    F.col("eviction_order_yes") / F.col("eviction_filings_completed") * 100))
    .withColumn("tenant_represented_pct", F.when(F.col("eviction_filings_completed") > 0,
                                    F.col("tenant_represented") / F.col("eviction_filings_completed") * 100))
)

eviction_cols = [
    "eviction_filings_total", "eviction_filings_rate", "eviction_rate_per_100", "eviction_filings_completed",
    "eviction_order_pct", "tenant_represented_pct", "back_rent_median",
]
census_cols = [
    "housing_units_rental", "median_rent", "poverty_rate", "pct_black", "pct_latinx", "pct_white", "renter_share",
]

print("=== Table 1: Data Overview ===")
print("Rows (tract-years):", lcbh.count())
print("Census tracts:", lcbh.select("tract_id").distinct().count())
year_range = lcbh.agg(F.min("year"), F.max("year")).first()
print("Years:", year_range[0], "to", year_range[1])
print("Filing counts originally missing:", lcbh.filter(F.col("count_missing")).count())
if "rental_missing" in lcbh.columns:
    print("Rental unit counts originally missing:", lcbh.filter(F.col("rental_missing")).count())
print("Rows with no filing rate (0 rental units):", lcbh.filter(F.col("eviction_rate_per_100").isNull()).count())
print("Rows with fewer than 100 rental units:", lcbh.filter(F.col("housing_units_rental") < 100).count())
print("Rows with no median rent reported:", lcbh.filter(F.col("median_rent").isNull()).count())
rate_corr = lcbh.agg(F.corr("eviction_filings_rate", "eviction_rate_per_100")).first()[0]
print("Correlation between LCBH's rate and our recomputed rate:", round(rate_corr, 3))
print("Rows per ACS period:")
display(lcbh.groupBy("acs_key").count().orderBy("acs_key").toPandas())

print("=== Table 2: Eviction Measures (all tract-years) ===")
rows = []
for c in eviction_cols:
    stats = lcbh.agg(
        F.count(c).alias("n"),
        F.mean(c).alias("mean"),
        F.stddev(c).alias("sd"),
        F.min(c).alias("min"),
        F.percentile_approx(c, 0.10).alias("p10"),
        F.percentile_approx(c, 0.50).alias("median"),
        F.percentile_approx(c, 0.90).alias("p90"),
        F.max(c).alias("max"),
        F.skewness(c).alias("skew"),
        F.kurtosis(c).alias("kurtosis"),
    ).first().asDict()
    stats["measure"] = c
    rows.append(stats)
eviction_summary = pd.DataFrame(rows)
eviction_summary = eviction_summary[["measure", "n", "mean", "sd", "min", "p10", "median", "p90", "max", "skew", "kurtosis"]]
display(eviction_summary.round(2))

print("=== Table 3: Filings by Year ===")
yearly = (
    lcbh.groupBy("year")
    .agg(
        F.sum("eviction_filings_total").alias("filings"),
        F.sum(F.when(F.col("housing_units_rental") > 0, F.col("eviction_filings_total"))).alias("filings_with_units"),
        F.sum("housing_units_rental").alias("rental_units"),
        F.mean("eviction_rate_per_100").alias("mean_tract_rate"),
        F.percentile_approx("eviction_rate_per_100", 0.5).alias("median_tract_rate"),
    )
    .orderBy("year")
    .toPandas()
)
yearly["citywide_rate"] = yearly["filings_with_units"] / yearly["rental_units"] * 100
yearly["change_from_last_year_pct"] = yearly["filings"].pct_change() * 100
display(yearly.round(2))

print("=== Table 4: Census Features by ACS Period (one row per tract) ===")
tract_acs = lcbh.select(["tract_id", "acs_key"] + census_cols).dropDuplicates(["tract_id", "acs_key"])
rows = []
for c in census_cols:
    results = (
        tract_acs.groupBy("acs_key")
        .agg(F.count(c).alias("n"), F.mean(c).alias("mean"), F.percentile_approx(c, 0.5).alias("median"))
        .orderBy("acs_key")
        .collect()
    )
    for r in results:
        rows.append({"feature": c, "acs_period": r["acs_key"], "n": r["n"], "mean": r["mean"], "median": r["median"]})
census_summary = pd.DataFrame(rows)
display(census_summary.round(2))

In [0]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")
RED = "#d32f2f"
GREEN = "#2e7d32"
NAMES = {
    "avg_rate": "Filing rate",
    "avg_filings": "Filings per year",
    "housing_units_rental": "Rental units",
    "median_rent": "Median rent ($)",
    "poverty_rate": "Poverty rate (%)",
    "pct_black": "Black residents (%)",
    "pct_latinx": "Latinx residents (%)",
    "pct_white": "White residents (%)",
    "renter_share": "Rental share of housing (%)",
}

panel = lcbh.select(
    "tract_id", "year", "eviction_filings_total", "housing_units_rental", "eviction_rate_per_100"
).toPandas()

tract_avg = panel.groupby("tract_id").mean(numeric_only=True)
tract_avg = tract_avg[tract_avg["housing_units_rental"] >= 100]
citywide_avg = yearly["filings_with_units"].sum() / yearly["rental_units"].sum() * 100

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Chicago Eviction Filings (LCBH), 2010-2019", fontsize=14, fontweight="bold")

axes[0].bar(yearly["year"], yearly["filings"] / 1000, color=GREEN)
axes[0].set_title("Eviction Filings per Year", fontweight="bold")
axes[0].set_ylabel("Filings (thousands)")
axes[0].set_xticks(yearly["year"])
axes[0].tick_params(axis="x", rotation=45)

axes[1].plot(yearly["year"], yearly["citywide_rate"], color=RED, marker="o", linewidth=2, label="Citywide rate")
axes[1].plot(yearly["year"], yearly["median_tract_rate"], color=GREEN, marker="s", linestyle="--", linewidth=2, label="Median tract")
axes[1].set_ylim(0, yearly["citywide_rate"].max() * 1.4)
axes[1].axvline(2012.5, color="gray", linestyle=":")
axes[1].set_ylabel("Filings per 100 rental units")
axes[1].set_xticks(yearly["year"])
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend(loc="lower left")

rates = tract_avg["eviction_rate_per_100"]
ratio = rates / citywide_avg
groups = ["Under half", "Half to 1x", "1x to 2x", "2x or more"]
counts = [
    (ratio < 0.5).sum(),
    ((ratio >= 0.5) & (ratio < 1)).sum(),
    ((ratio >= 1) & (ratio < 2)).sum(),
    (ratio >= 2).sum(),
]
share_above = (ratio >= 1).mean() * 100
share_double = (ratio >= 2).mean() * 100
bars = axes[2].bar(groups, counts, color=[GREEN, GREEN, RED, RED])
axes[2].bar_label(bars, labels=[f"{c / len(rates) * 100:.0f}%" for c in counts], padding=3)
axes[2].set_ylim(0, max(counts) * 1.15)
axes[2].set_title(f"{share_double:.0f}% of Tracts Have Double the Citywide Rate", fontweight="bold")
axes[2].set_xlabel("Tract's 10-year average compared with the citywide average")
axes[2].set_ylabel("Number of tracts")

plt.tight_layout()
plt.show()

tract_aggregations = [
    F.mean("eviction_rate_per_100").alias("avg_rate"),
    F.mean("eviction_filings_total").alias("avg_filings"),
]

for c in census_cols:
    tract_aggregations.append(F.first(c).alias(c))
tract_era = lcbh.groupBy("tract_id", "acs_key").agg(*tract_aggregations).toPandas()
tract_era = tract_era[tract_era["housing_units_rental"] >= 100]
periods = sorted(tract_era["acs_key"].unique())

fig2_features = ["median_rent", "poverty_rate", "pct_black", "pct_latinx"]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle("Mean Filing Rate by Neighborhood Quartile", fontsize=14, fontweight="bold")
quartile_rows = []
for ax, feature in zip(axes.flatten(), fig2_features):
    for i, period in enumerate(periods):
        sub = tract_era[tract_era["acs_key"] == period].dropna(subset=[feature, "avg_rate"]).copy()
        sub["quartile"] = pd.qcut(sub[feature].rank(method="first"), 4, labels=["Q1", "Q2", "Q3", "Q4"])
        means = sub.groupby("quartile", observed=True)["avg_rate"].mean()
        rho = sub[feature].corr(sub["avg_rate"], method="spearman")
        x = np.arange(4) + (i - 0.5) * 0.4
        ax.bar(x, means.values, width=0.4, color=[RED, GREEN][i], label=f"ACS {period} (Spearman = {rho:.2f})")
        quartile_rows.append({"feature": feature, "acs_period": period, "Q1_rate": means.iloc[0],
                              "Q4_rate": means.iloc[3], "Q4_vs_Q1": means.iloc[3] / means.iloc[0], "spearman": rho})
    ax.set_xticks(range(4))
    ax.set_xticklabels(["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"])
    ax.set_title(NAMES[feature], fontweight="bold")
    ax.set_ylabel("Mean filings per 100 rental units")
    ax.set_ylim(0, ax.get_ylim()[1] * 1.3)
    ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

latest = tract_era[tract_era["acs_key"] == periods[-1]]
features = ["median_rent", "poverty_rate", "pct_black", "pct_latinx", "pct_white", "renter_share"]
correlations = []
for c in features:
    correlations.append(latest[c].corr(latest["avg_rate"], method="spearman"))
corr_table = pd.DataFrame({"feature": [NAMES[c] for c in features], "correlation": correlations})
corr_table = corr_table.sort_values("correlation")

comm = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(DATA_FOLDER + "/eviction_data_comm_area.csv")
)

area_avg = comm.groupBy("area_name").agg(F.mean("eviction_filings_rate").alias("avg_rate")).toPandas()
top_areas = area_avg.sort_values("avg_rate", ascending=False).head(15)
top_areas = top_areas.iloc[::-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("What Goes With High Eviction Rates, and Where They Are", fontsize=14, fontweight="bold")

colors = [RED if v > 0 else GREEN for v in corr_table["correlation"]]
bars = axes[0].barh(corr_table["feature"], corr_table["correlation"], color=colors)
axes[0].bar_label(bars, fmt="%+.2f", padding=3)
axes[0].axvline(0, color="black", linewidth=1)
axes[0].set_xlim(-1, 1)
axes[0].set_title(f"Neighborhood Features vs. Filing Rate (ACS {periods[-1]})", fontweight="bold")
axes[0].set_xlabel("Spearman correlation (red = more evictions, green = fewer)")

bars = axes[1].barh(top_areas["area_name"], top_areas["avg_rate"], color=RED)
axes[1].bar_label(bars, labels=[f"{r / citywide_avg:.1f}x city" for r in top_areas["avg_rate"]], padding=3)
axes[1].axvline(citywide_avg, color=GREEN, linestyle="--", linewidth=2, label=f"Citywide average = {citywide_avg:.2f}")
axes[1].set_xlim(0, top_areas["avg_rate"].max() * 1.25)
axes[1].set_title("15 Community Areas With the Highest Filing Rates, 2010-2019", fontweight="bold")
axes[1].set_xlabel("Average filings per 100 rental units")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

In [0]:
first = yearly.iloc[0]
last = yearly.iloc[-1]
peak = yearly.loc[yearly["filings"].idxmax()]
change = (last["filings"] - first["filings"]) / first["filings"] * 100

print(f"Filings went from {first['filings']:,.0f} in {first['year']:.0f} to {last['filings']:,.0f} in {last['year']:.0f} ({change:+.1f}%).")
print(f"The peak year was {peak['year']:.0f} with {peak['filings']:,.0f} filings.")
print(f"Citywide rate went from {first['citywide_rate']:.2f} to {last['citywide_rate']:.2f} filings per 100 rental units.")
print(f"Citywide average for 2010-2019: {citywide_avg:.2f} (LCBH's portal reports 3.7).")
print(f"{share_above:.0f}% of tracts are above the citywide average and {share_double:.0f}% are at least double.")

print("\nFiling rate by quartile (Figure 2):")
display(pd.DataFrame(quartile_rows).round(2))

print("\nCorrelation with the tract filing rate")
display(corr_table.round(2))

print("\nHighest-rate community areas")
display(top_areas.iloc[::-1].round(2))